# 🎥 中文人像口播视频生成器

上传一张人像照片 + 填写中文文案，即可生成带有配音与嘴型同步的视频。

## 🔧 技术组件
- 中文语音合成：Edge-TTS 或 Bark
- 嘴型同步：SadTalker

In [ ]:
# ✅ 安装必要依赖
!pip install edge-tts gradio git+https://github.com/OpenTalker/SadTalker.git
!pip install git+https://github.com/suno-ai/bark.git
!pip install imageio[ffmpeg] moviepy


In [ ]:
# 📤 上传图像
from google.colab import files
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

In [ ]:
# 📝 输入中文口播文案
text = input("请输入中文口播文案：")
# ✅ 配音方式选择
use_bark = False  # True 使用 Bark，False 使用 Edge-TTS

In [ ]:
# 🔊 合成语音
if use_bark:
    from bark import SAMPLE_RATE, generate_audio, preload_models
    preload_models()
    audio_array = generate_audio(text)
    from scipy.io.wavfile import write as write_wav
    write_wav("output.wav", SAMPLE_RATE, audio_array)
else:
    import edge_tts
    import asyncio
    async def edge_speak():
        communicate = edge_tts.Communicate(text, voice="zh-CN-XiaoxiaoNeural")
        await communicate.save("output.wav")
    asyncio.run(edge_speak())

In [ ]:
# 🤖 生成嘴型同步视频（SadTalker）
!mkdir -p inputs
!cp "$image_path" inputs/face.png
!python -m sadtalker --driven_audio output.wav --source_image inputs/face.png --result_dir results --still --preprocess full --enhancer gfpgan

In [ ]:
# 🎬 播放视频
from IPython.display import Video
Video("results/results.mp4", embed=True)

In [ ]:
# ⬇️ 提供下载
files.download("results/results.mp4")